# Thesis Baseline Setup

This notebook documents the workflow for the thesis baseline on xBD/xView2 using a building-level classification pipeline.

---

## Core Principle

- Preprocessing and model training run directly in Python
- Notebook is used for:
  - Data inspection
  - Sanity checks
  - Training control
  - Evaluation
  - Visualization

The official xView2 Docker pipeline is not used as the main experimental setup, because the thesis isolates the damage classification task from localization by using ground truth building polygons.

---

## Required Data

Download and place on Desktop:

- `train`
- `test`
- `hold`

These contain:
- images
- labels

---

## Main Experimental Design

The thesis does not use the original end to end competition pipeline as the primary method.

Instead, the pipeline is:

1. Read ground truth polygons from label JSON files
2. Build a building-level metadata table
3. Extract pre/post building crops
4. Stack pre and post crops into 6-channel tensors
5. Train a ResNet50-based classifier on damage labels

This removes localization as a confounding factor and isolates the classification task.

---

## Required External Apps

- `miniconda` or another Python environment manager
- optionally `docker` only for reproducing the original released baseline, not for the main thesis experiments

---

## Create Environment

```bash
conda create -n thesis_xview python=3.10
conda activate thesis_xview

In [1]:
import sys
!{sys.executable} -m pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
  Using cached libtiff-0.4.2.tar.gz (129 kB)
     |████████████████████████████████| 948 kB 390 kB/s eta 0:00:01
     |████████████████████████████████| 323 kB 469 kB/s eta 0:00:01
     |████████████████████████████████| 1.4 MB 2.7 MB/s eta 0:00:01
  Using cached imantics-0.1.12-py3-none-any.whl
     |████████████████████████████████| 1.4 MB 1.1 MB/s eta 0:00:01
  distutils: /private/var/folders/bs/rzk6qg1902vgdlp6h22t8bq40000gn/T/pip-build-env-42t6_521/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  distutils: /private/var/folders/bs/rzk6qg1902vgdlp6h22t8bq40000gn/T/pip-build-env-42t6_521/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  user = False
  home = None
  root = None
  prefix = '/private/var/folders/bs/rzk6qg1902vgdlp6h22t8bq40000gn/T/pip-build-env-42t6_521/normal'
  distutils: /private/var/folders/bs/rzk6qg1902vgdlp6h22t8bq4000

In [3]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 73.6 MB 1.2 MB/s eta 0:00:011     |████████████████████████████    | 64.6 MB 15.3 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 1.3 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 12.4 MB/s eta 0:00:01
     |████████████████████████████████| 6.3 MB 896 kB/s eta 0:00:011
     |████████████████████████████████| 134 kB 885 kB/s eta 0:00:01
     |████████████████████████████████| 200 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 536 kB 689 kB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import torch
import numpy as np
import pandas as pd
import cv2
import shapely

print("All packages loaded successfully")
print("Torch version:", torch.__version__)

All packages loaded successfully
Torch version: 2.8.0


In [8]:
import sys
!{sys.executable} model/classification_training.py

Loading data...

Split sizes:
split
train    159793
test      53850
hold      53137
Name: count, dtype: int64

Train label distribution:
damage_label
no-damage       117425
minor-damage     14980
major-damage     14161
destroyed        13227
Name: count, dtype: int64

Using device: cpu

Using class weights: [0.34020224 2.6667724  2.8210049  3.0202048 ]

Starting training...
^C
  File "/Users/paolo/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/module.py", line 1784, in _call_impl
    return forward_call(*args, **kwargs)
  File "/Users/paolo/Library/Python/3.9/lib/python/site-packages/torchvision/models/resnet.py", line 285, in forward
    return self._forward_impl(x)
  File "/Users/paolo/Library/Python/3.9/lib/python/site-packages/torchvision/models/resnet.py", line 275, in _forward_impl
    x = self.layer3(x)
  File "/Users/paolo/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/module.py", line 1773, in _wrapped_call_impl
    return self._call_impl(*args, **k